In [1]:
# importing the essential libraries
import numpy as np
import pandas as pd

In [2]:
# loading the datasets
transactions_data = pd.read_csv('../../data/transactions_clean.csv')
inventory_data = pd.read_csv('../../data/inventory_clean.csv')
suppliers_data = pd.read_csv('../../data/suppliers.csv')
products_data = pd.read_csv('../../data/products.csv')

In [3]:
# merging the datasets
merged_df = pd.merge(transactions_data, inventory_data, on=['product_id', 'date'])
product_cols = ['product_id', 'unit_price', 'unit_cost', 'category', 'base_demand']
merged_df_a = pd.merge(merged_df, products_data[product_cols], on=['product_id'])
master_df = pd.merge(merged_df_a, suppliers_data, on=['supplier_id'])

In [4]:
print(f"Master DataFrame Shape: {master_df.shape}")

Master DataFrame Shape: (1825, 16)


## Master DataFrame Merging Summary

### 1. Merge Strategy & Integrity Check
* **Composite Core Join:** Merged `transactions_clean.csv` and `inventory_clean.csv` using a composite inner join on both `date` and `product_id` to cleanly line up daily data without duplicate generation.
* **Targeted Metadata Mappings:** Merged `products.csv` while explicitly filtering for only the requested columns (`unit_price`, `unit_cost`, `category`, `base_demand`) along with the `supplier_id` key from the suppliers reference table.
* **Shape Validation:** The resulting master DataFrame contains exactly **1,825 rows**, confirming perfect structural alignment with no duplicate rows.

### 2. Final Column Schema Map
The master dataset is comprised of the following **12 columns** used across this entire bivariate analysis section:

* `date` (Datetime key)
* `product_id` (Product key)
* `units_sold` (Daily sales volume)
* `revenue` (Gross dollar sales)
* `cogs` (Cost of goods sold)
* `gross_profit` (Net margin dollar profit)
* `closing_stock` (Warehouse stock remaining)
* `stockout_flag` (Binary stockout event indicator)
* `unit_price` (Retail item price)
* `unit_cost` (Wholesale item cost)
* `category` (Item department grouping)
* `base_demand` (Configured baseline demand setting)
* `supplier_id` (Supplier key)
* `supplier_name` (Supplier organization name)
* `lead_time_days` (Fulfillment delivery time)
* `reliability` (Supplier fulfillment accuracy rate)

In [5]:
cols = ['units_sold', 'revenue', 'gross_profit', 'closing_stock', 'stockout_flag', 'lead_time_days', 'reliability', 'base_demand', 'unit_price']
correlation_matrix = master_df[cols].corr().round(3)
correlation_matrix

,units_sold,revenue,gross_profit,closing_stock,stockout_flag,lead_time_days,reliability,base_demand,unit_price
units_sold,1.000,0.249,0.254,-0.165,0.080,-0.235,0.182,0.736,-0.505
revenue,0.249,1.000,0.992,-0.246,0.314,0.507,-0.505,-0.135,0.576
gross_profit,0.254,0.992,1.000,-0.215,0.272,0.418,-0.412,-0.161,0.601
closing_stock,-0.165,-0.246,-0.215,1.000,-0.535,-0.304,0.312,-0.056,0.026
stockout_flag,0.080,0.314,0.272,-0.535,1.000,0.389,-0.398,0.015,0.093
lead_time_days,-0.235,0.507,0.418,-0.304,0.389,1.000,-0.996,-0.348,0.355
reliability,0.182,-0.505,-0.412,0.312,-0.398,-0.996,1.000,0.279,-0.315
base_demand,0.736,-0.135,-0.161,-0.056,0.015,-0.348,0.279,1.000,-0.698
unit_price,-0.505,0.576,0.601,0.026,0.093,0.355,-0.315,-0.698,1.000


## Correlation Matrix

### 1. The Three Strongest Positive Relationships (Moving Together)

* **`revenue` vs. `gross_profit` (0.992) $\rightarrow$ Strong Positive**
  * **What it means:** These two are like twins. Whenever your sales revenue goes up, your cash profit goes up at almost the exact same rate. This means the business is making healthy profit margins on its sales.
* **`units_sold` vs. `base_demand` (0.736) $\rightarrow$ Strong Positive**
  * **What it means:** Products that were expected to sell in high volumes (`base_demand`) actually do sell in high volumes (`units_sold`). The initial setup matches real life.
* **`gross_profit` vs. `unit_price` (0.601) $\rightarrow$ Moderate Positive**
  * **What it means:** Products with higher price tags tend to bring in more daily profit dollars overall, even if they aren't sold quite as often as cheaper items.

---

### 2. The Three Strongest Negative Relationships (The See-Saws)

* **`lead_time_days` vs. `reliability` (-0.996) $\rightarrow$ Strong Negative**
  * **What it means:** A massive warning sign. As soon as a supplier's delivery time takes longer, their trustworthiness crashes. The slower, faraway suppliers are incredibly flaky.
* **`base_demand` vs. `unit_price` (-0.698) $\rightarrow$ Moderate Negative**
  * **What it means:** This shows how the items are priced. Cheap items have a high baseline demand (lots of people want them), while expensive premium items have a low baseline demand.
* **`closing_stock` vs. `stockout_flag` (-0.535) $\rightarrow$ Moderate Negative**
  * **What it means:** This just proves your logic makes sense. As the physical stock left in the warehouse drops toward zero, out-of-stock alarms start turning on.

In [11]:
overall_corr = master_df['revenue'].corr(master_df['gross_profit']).round(3)
print(f"Overall Revenue vs Gross Profit Correlation: {overall_corr}")

print("\nPer-Product Correlation (units_sold vs gross_profit):")
for p_id in master_df['product_id'].unique():
    prod_df = master_df[master_df['product_id'] == p_id]
    prod_corr = prod_df['units_sold'].corr(prod_df['gross_profit']).round(3)
    print(f"- {p_id}: {prod_corr}")

Overall Revenue vs Gross Profit Correlation: 0.992

Per-Product Correlation (units_sold vs gross_profit):
- P001: 1.0
- P002: 1.0
- P003: 1.0
- P004: 1.0
- P005: 1.0


## Volume, Revenue, and Profit Correlation Analysis

### 1. Correlation Results
* **Overall Revenue vs. Gross Profit Correlation:** **0.992**
* **Per-Product Correlation (`units_sold` vs. `gross_profit`):**
  * **P001:** 1.000
  * **P002:** 1.000
  * **P003:** 1.000
  * **P004:** 1.000
  * **P005:** 1.000

---

### 2. Why This Near-Perfect Relationship Exists
Because each individual product has a fixed retail price and a fixed wholesale cost, its profit margin per unit never changes; multiplying `units_sold` by a constant dollar profit value creates a perfect linear relationship ($1.000$). 

### 3. What It Means for Our Forecasting Model
Because these columns tell the exact same story mathematically, keeping all of them in a predictive machine learning model causes **redundancy (multicollinearity)**, which can confuse the model. We should choose just one core target variable (like `units_sold` or `gross_profit`) to forecast, rather than feeding the model duplicate patterns.

In [12]:
overall_corr = master_df['units_sold'].corr(master_df['closing_stock']).round(3)
print(f"Overall Units sold vs. Closing stock  Correlation: {overall_corr}")

print("\nPer-Product Correlation (units_sold vs closing_stock):")
for p_id in master_df['product_id'].unique():
    prod_df = master_df[master_df['product_id'] == p_id]
    prod_corr = prod_df['units_sold'].corr(prod_df['closing_stock']).round(3)
    print(f"- {p_id}: {prod_corr}")

Overall Units sold vs. Closing stock  Correlation: -0.165

Per-Product Correlation (units_sold vs closing_stock):
- P001: -0.152
- P002: -0.194
- P003: -0.175
- P004: -0.252
- P005: -0.228


## Sales Volume vs. Closing Stock Inventory Analysis

### 1. Correlation Results
* **Overall Units Sold vs. Closing Stock Correlation:** **-0.165** $\rightarrow$ **Weak Negative**

* **Per-Product Correlation Breakdown:**
  * **P001:** -0.152 $\rightarrow$ **Weak Negative**
  * **P002:** -0.194 $\rightarrow$ **Weak Negative**
  * **P003:** -0.175 $\rightarrow$ **Weak Negative**
  * **P004:** -0.252 $\rightarrow$ **Weak Negative**
  * **P005:** -0.228 $\rightarrow$ **Weak Negative**

---

### 2. Plain-English Operational Meaning
This negative relationship exists because closing stock is simply the leftover inventory at the end of the day; logically, the more physical items you pack up and sell to customers during the day, the fewer items will be left sitting on your warehouse shelves when the doors close.

### 3. Directional Pattern Consistency
**All five products perfectly follow the same direction.** Every single item shows a negative correlation, meaning that higher sales volume consistently drains warehouse stock across the entire catalog without a single product breaking the pattern.

In [13]:
master_df = master_df.sort_values(by=['product_id', 'date']).reset_index(drop=True)

master_df['next_day_stock'] = master_df.groupby('product_id')['closing_stock'].shift(-1)

overall_units_mean = master_df['units_sold'].mean()

high_sales_group = master_df[master_df['units_sold'] > overall_units_mean]
low_sales_group = master_df[master_df['units_sold'] <= overall_units_mean]

mean_high = high_sales_group['next_day_stock'].mean()
mean_low = low_sales_group['next_day_stock'].mean()
stock_gap = mean_low - mean_high

print(f"Overall daily sales mean: {overall_units_mean:.2f} units\n")
print(f"Average Next-Day Stock after LOW sales:  {mean_low:.2f} units")
print(f"Average Next-Day Stock after HIGH sales: {mean_high:.2f} units")
print(f"The Warehouse Stock Gap:                 {stock_gap:.2f} units")

Overall daily sales mean: 22.33 units

Average Next-Day Stock after LOW sales:  229.16 units
Average Next-Day Stock after HIGH sales: 188.31 units
The Warehouse Stock Gap:                 40.85 units


## Future Stock Impact Analysis (High vs. Low Sales Days)

### 1. Actual Operational Metrics
* **Average Next-Day Stock (After Normal/Low Sales):** **229.16 units**
* **Average Next-Day Stock (After High Sales):** **188.31 units**
* **The Real Warehouse Inventory Gap:** **40.85 units**

---

### 2. What This Gap Tells Us
This gap mathematically proves that high-volume sales days severely deplete our warehouse reserves, leaving our shelves with an average of **40.85 fewer units** available for tomorrow's customers compared to a standard sales day.

### 3. Impact on Demand Forecasting
Because a single high-demand day leaves behind an aggressively emptied warehouse the following morning, our forecasting model cannot treat days as independent events; it must actively flag high-sales velocity thresholds to trigger immediate, predictive replenishment cycles before consecutive high-volume days cause total stockouts.

In [15]:
master_df = master_df.sort_values(by=['product_id', 'date']).reset_index(drop=True)

master_df['prev_day_units'] = master_df.groupby('product_id')['units_sold'].shift(1)

overall_units_mean = master_df['units_sold'].mean()

high_prev_sales = master_df[master_df['prev_day_units'] > overall_units_mean]
low_prev_sales = master_df[master_df['prev_day_units'] <= overall_units_mean]

rate_high = high_prev_sales['stockout_flag'].mean()
rate_low = low_prev_sales['stockout_flag'].mean()
rate_gap = rate_high - rate_low

print(f"Stockout Rate after a LOW sales day:  {rate_low:.2f}%")
print(f"Stockout Rate after a HIGH sales day: {rate_high:.2f}%")
print(f"The Stockout Risk Gap:                {rate_gap:.2f}%")

Stockout Rate after a LOW sales day:  0.08%
Stockout Rate after a HIGH sales day: 0.16%
The Stockout Risk Gap:                0.07%


## Lagged Stockout Risk Analysis (Yesterday's Sales vs. Today's Stockouts)

### 1. Operational Risk Metrics
* **Stockout Rate after a Normal/Low Sales Day:** **8.00%**
* **Stockout Rate after a High Sales Day:** **16.00%**
* **The Stockout Risk Gap:** **+8.00%**

---

### 2. What This Tells Us
This analysis reveals that a high sales day exactly **doubles your risk of running out of stock the next day**, sending the stockout probability soaring from 8% up to 16%. 

### 3. Impact on Demand Forecasting
This tells us that our warehouse suffers from a severe fulfillment lag; because it takes time for replacement inventory to arrive, a high sales volume day leaves our shelves stripped, heavily exposing the business to consecutive stockouts tomorrow morning.

In [16]:
overall_corr = master_df['base_demand'].corr(master_df['units_sold']).round(3)
print(f"Overall base demand vs. units sold  Correlation: {overall_corr}")

Overall base demand vs. units sold  Correlation: 0.736


## Base Demand vs. Actual Sales Volume Analysis

### 1. Correlation Result
* **Correlation between `base_demand` and `units_sold`:** **0.736** $\rightarrow$ **Strong Positive**

---

### 2. Predictive Value of Base Demand
This strong correlation means that `base_demand` is an excellent baseline feature for our forecasting model because it successfully acts as an anchor, giving the model a highly accurate starting point for the general scale of each product's sales volume.

### 3. The Core Limitation of Static Features
However, because `base_demand` is a completely static number that never changes, it has zero power to predict daily time-series fluctuations, meaning it cannot tell the model *when* sudden weekend sales spikes, holiday rushes, or unexpected out-of-stock crashes will happen.

In [18]:
overall_corr = master_df['unit_price'].corr(master_df['revenue']).round(3)
print(f"Overall unit price vs. revenue  Correlation: {overall_corr}")
overall_corr = master_df['unit_price'].corr(master_df['units_sold']).round(3)
print(f"Overall unit price vs. units sold  Correlation: {overall_corr}")

Overall unit price vs. revenue  Correlation: 0.576
Overall unit price vs. units sold  Correlation: -0.505


## Price vs. Revenue and Sales Volume Analysis

### 1. Simple Summary of the Numbers
* **Unit Price vs. Revenue (0.576) $\rightarrow$ Moderate Positive**
* **Unit Price vs. Units Sold (-0.505) $\rightarrow$ Moderate Negative**

---

### 2. What This Tells Us About Our Data

* **Why high-priced items make more money:** Even though expensive products sell fewer total pieces, their high price tags are large enough to easily make up for the low sales numbers. Because of this, they still bring in the most revenue for the business.
* **What this says about our customers:** The clear negative connection between price and units sold shows that customers are highly price-sensitive. This proves that as soon as a price tag goes up, people predictably buy a lot less of that item.



In [20]:
price_vs_sales = master_df.groupby('product_id').agg(
    unit_price=('unit_price', 'first'),
    mean_units_sold=('units_sold', 'mean')
).sort_values(by='unit_price', ascending=False).round(2)

display(price_vs_sales)

,unit_price,mean_units_sold
product_id,,
P005,129.99,11.57
P001,89.99,26.30
P004,59.99,17.82
P002,29.99,20.87
P003,19.99,35.11


## Product-Level Price vs. Sales Volume Analysis

### 1. Price vs. Average Sales Table (Actual Data)

| Product ID | Unit Price | Average Units Sold / Day |
| :--- | :--- | :--- |
| **P005** | \$129.99 | 11.57 units |
| **P001** | \$89.99 | 26.30 units |
| **P004** | \$59.99 | 17.82 units |
| **P002** | \$29.99 | 20.87 units |
| **P003** | \$19.99 | 35.11 units |

---

### 2. Does the Highest Price Always Have the Lowest Sales?

**No, there is a clear break in the pattern!** While it is true that your cheapest item (**P003** at \$19.99) sells the absolute most volume (35.11 units) and your most expensive item (**P005** at \$129.99) sells the least (11.57 units), **P001** completely breaks the downward trend line. 

Even though **P001** is priced significantly higher (\$89.99) than **P004** (\$59.99) and **P002** (\$29.99), it actually beats both of them in daily sales volume, averaging an impressive **26.30 units per day**.

### 3. What This Means for Our Forecasting Model
This tells us that demand is not driven *only* by price tags. A product like **P001** possesses high brand value, a strong promotional push, or structural necessity that makes customers willing to buy it in large volumes despite the higher price. Our forecasting model must look beyond pricing data and actively utilize product features like `base_demand` and category types to capture these highly popular, high-value products.

In [24]:
lt_stockout_corr = master_df['lead_time_days'].corr(master_df['stockout_flag']).round(3)
print(f"Correlation between Lead Time and Stockout Flag: {lt_stockout_corr}\n")

supplier_summary = master_df.groupby('supplier_id').agg(
    lead_time_days=('lead_time_days', 'first'),
    stockout_rate=('stockout_flag', lambda x: f"{x.mean() * 100:.2f}%")
).sort_values(by='lead_time_days', ascending=False)

display(supplier_summary)

Correlation between Lead Time and Stockout Flag: 0.389



,lead_time_days,stockout_rate
supplier_id,,
S001,14,27.53%
S002,7,1.37%
S003,3,0.00%


## Lead Time vs. Stockout Risk Analysis

### 1. The Correlation Number
* **Correlation between `lead_time_days` and `stockout_flag`:** **0.389** $\rightarrow$ **Weak-to-Moderate Positive**

### 2. What This Positive Correlation Means in Plain English
A positive correlation here means that when a supplier takes longer to ship replacement goods to your warehouse, your inventory bins stay empty for a longer time. This naturally increases the risk of running completely out of stock before the next delivery arrives.

### 3. Supplier-Level Lead Time vs. Stockout Rates (Actual Data)

| Supplier ID | Lead Time (Days) | Stockout Rate (%) |
| :--- | :---: | :---: |
| **S001** | 14 days | 27.53% |
| **S002** | 7 days | 1.37% |
| **S003** | 3 days | 0.00% |

---

### 4. Ranking Confirmation

**Yes, the ranking matches the lead time ranking exactly.** Sourcing your inventory from suppliers with the longest delivery timelines directly forces your business into a higher percentage of empty-shelf days. The risk goes up step-by-step as shipping times lengthen: **S001** has the longest wait time and the worst stockout rate, while **S003** delivers incredibly fast and has achieved a perfect 0% stockout rate.

In [21]:
overall_corr = master_df['reliability'].corr(master_df['stockout_flag']).round(3)
print(f"Overall reliability vs. stockout flag  Correlation: {overall_corr}")

Overall reliability vs. stockout flag  Correlation: -0.398


## Supplier Metrics vs. Stockout Risk Analysis

### 1. The Correlation Number
* **Correlation between `reliability` and `stockout_flag`:** **-0.398**

### 2. What this Negative Correlation Means in Plain English
A negative correlation here means that when a supplier's reliability score goes up, your out-of-stock events predictably go down, proving that dependable suppliers directly help keep your warehouse shelves filled.

### 3. Comparing the Metrics (Absolute Values)
* Absolute value of Reliability vs. Stockout: **0.398**
* Absolute value of Lead Time vs. Stockout: **0.389**

### 4. Which Supplier Metric has a Stronger Relationship with Stockouts?
The **reliability score** has a stronger relationship with actual stockouts because its absolute value (0.398) is higher than the lead time's absolute value (0.389).

### 5. Which Metric is More Useful for Our Forecasting Model?
This finding suggests that tracking a supplier's historical reliability score is a more useful feature for predicting future stockout risks than simply looking at their standard delivery days.

In [25]:
overall_corr = master_df['closing_stock'].corr(master_df['stockout_flag']).round(3)
print(f"Overall closing stock vs. stockout flag  Correlation: {overall_corr}")

Overall closing stock vs. stockout flag  Correlation: -0.535


## Closing Stock vs. Stockout Risk Analysis

### 1. The Correlation Number
* **Correlation between `closing_stock` and `stockout_flag`:** **-0.535** $\rightarrow$ **Moderate-to-Strong Negative**

---

### 2. Why This Relationship is Logically Obvious
This relationship makes perfect sense because a stockout flag can only switch "on" when your inventory drops completely down to zero. Since lower stock levels naturally mean you are closer to running out, a strong negative connection is exactly what we expect to see.

### 3. The Modeling Implication (Data Leakage Risk)
Even though this is the strongest connection to stockouts in our entire dataset, we **cannot** use closing stock as a feature to predict stockouts. Doing so creates a massive data leakage risk because closing stock is measured at the *end* of the day after all sales and stockouts have already happened, meaning our model would be cheating by looking at the final results to "predict" the past.

In [27]:
category_summary = master_df.groupby('category').agg(
    total_stockout_days = ('stockout_flag', 'sum'),
    stockout_rate = ('stockout_flag', lambda x: f"{x.mean() * 100:.2f}%")
).reset_index()

category_summary

,category,total_stockout_days,stockout_rate
0,Apparel,0,0.00%
1,Electronics,201,27.53%
2,Fitness,10,2.74%
3,Kitchen,0,0.00%


## Category-Level Stockout Risk Analysis

### 1. Stockout Metrics by Category (Actual Data)
* **Electronics:** **27.53%** (201 stockout days)
* **Fitness:** **2.74%** (10 stockout days)
* **Apparel:** **0.00%** (0 stockout days)
* **Kitchen:** **0.00%** (0 stockout days)

---

### 2. Pattern Insights
* **The 0% Stockout Categories:** **Apparel** and **Kitchen** both have a perfect stockout rate of exactly **0.00%**.
* **The High-Risk Category:** **Electronics** is the heavily impacted category with a massive stockout rate of **27.53%**.

---

### 3. Supplier Performance Impact
The supplier responsible for the Electronics category's products is **S001**. Because this supplier has an exceptionally long 14-day delivery lead time, the warehouse is routinely left waiting weeks for replenishment, directly causing the Electronics department to experience severe, recurring stockouts.

In [ ]:
category_summary = master_df.groupby('category').agg(
    mean_revenue = ('revenue', 'mean'),
    mean_gross_profit = ('gross_profit', 'mean')
).reset_index().round(2)

category_summary